<!-- SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->


# 🔐 NeMo Safe Synthesizer Tutorial: Differential Privacy

Learn how to apply differential privacy to achieve the maximum level of privacy with mathematical guarantees. This tutorial demonstrates how to configure differential privacy parameters for optimal results. The runtime of this notebook is about 1 hour on an A100.

If you have not yet completed the [Safe Synthesizer 101](safe-synthesizer-101.ipynb) tutorial, consider starting there first.

### 🖥️ Prerequisites

This notebook requires a GPU. We recommend an H100; minimum A100.

### ⚡ Install Safe Synthesizer

Run the cell below to install NeMo Safe Synthesizer (engine and CUDA 12.8) and kagglehub for the example dataset.

In [ ]:
%%capture
!uv pip install nemo-safe-synthesizer[engine,cu128] --extra-index-url "https://urm.nvidia.com/artifactory/api/pypi/nv-shared-pypi-local/simple"
!uv pip install kagglehub


### 🔑 Set the inference API key for PII column classification

NeMo Safe Synthesizer uses an LLM‑based column classifier to automatically infer PII columns. To enable this feature, set `NSS_INFERENCE_KEY` (the inference endpoint defaults to the NVIDIA integrate URL. You can obtain an API key from [build.nvidia.com](https://build.nvidia.com/settings/api-keys)). Setting this value is optional but strongly recommended.

In [ ]:
import os
import getpass

# Setting NSS_INFERENCE_KEY is optional but strongly recommended for PII replacement.
if "NSS_INFERENCE_KEY" not in os.environ:
    os.environ["NSS_INFERENCE_KEY"] = getpass.getpass("Paste inference API key (or press Enter to skip): ")
if os.environ.get("NSS_INFERENCE_KEY"):
    print("NSS_INFERENCE_KEY is set")
else:
    print(
        "NSS_INFERENCE_KEY is not set. Replace PII will run in degraded mode. "
        "We strongly recommend setting a key."
    )

### 📥 Load and preview sample dataset

Load a tabular dataset—in this example, the [US Accidents dataset](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents) from Kaggle—and preview the first few rows. NeMo Safe Synthesizer will use a subset to keep runtime manageable.

This dataset includes text, categorical, and numeric fields, making it a good demonstration of the model's ability to handle multiple data types.

The code below also computes a recommended `delta` for differential privacy. Delta should reflect the full dataset size, not the subset, because it bounds the probability of a privacy breach across the entire population. See [Differential Privacy](../user-guide/configuration.md#differential-privacy) for parameter guidance.

In [ ]:
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("sobhanmoosavi/us-accidents")
print("Path to dataset files:", path)
df = pd.read_csv(f"{path}/US_Accidents_March23.csv", index_col=0)
full_data_size = len(df)
recommended_delta = 1 / (full_data_size ** 2)  # delta should reflect the full dataset, even when a subset is used as Safe Synthesizer input

print(f"Full dataset size: {len(df)} records")
print(f"Recommended delta: {recommended_delta:.2e}")

In [ ]:
# use a subset as Safe Synthesizer input for faster runtime
df = df.sample(n=26250, random_state=318)
print(f"Input dataset size: {len(df)} records")
df.head()

### ⚙️ Create and run Safe Synthesizer job

Create the Safe Synthesizer builder and attach your DataFrame. Enable differential privacy and configure the training and generation stages for optimal performance with DP.

Run the pipeline with `run()`, which performs data processing, PII replacement, training, generation, evaluation and saving of results in a single call. Results are available on `builder.results`.

In [ ]:
from nemo_safe_synthesizer.sdk.library_builder import SafeSynthesizer

builder = (
    SafeSynthesizer()
    .with_data_source(df)  # .with_replace_pii(enable=False) to disable PII replacement
    .with_differential_privacy(dp_enabled=True, delta=recommended_delta)
    .with_train(batch_size=16)  # Override the default batch size of 1, which is designed for non-DP training
    .with_generate(use_structured_generation=True)  # Improves the percentage of valid records when DP is enabled
)
builder.run()
results = builder.results

### 📤 Retrieve synthetic data

Inspect the generated synthetic data including row count and preview of the first rows.

In [ ]:
synth = results.synthetic_data
print(f"Number of synthetic rows: {len(synth)}")
synth.head()

In [ ]:
# Synthetic data and evaluation report are automatically saved to the artifacts directory
print(f"Artifacts automatically saved to: {builder._workdir.generate.path}")

### 🛡️ Review evaluation report

The pipeline computes both quality and privacy metrics. The summary includes timing information and overall scores, while the full evaluation report is rendered as an HTML document.

In [ ]:
import json

print("Summary (timing and scores):")
print(json.dumps(results.summary.model_dump(), indent=2))

In [ ]:
# View the evaluation report in a sandboxed iframe
import base64
from IPython.display import IFrame

report_html = results.evaluation_report_html
if report_html:
    data_url = "data:text/html;base64," + base64.b64encode(report_html.encode()).decode()
    display(IFrame(src=data_url, width="100%", height=800))